In [1]:

import pandas as pd

# Read the CSV file in chunks of 1000 rows
df_iter = pd.read_csv('../data/complaints.csv', chunksize=1000)

# Combine all chunks into one DataFrame
df = pd.concat(df_iter, ignore_index=True)

# Now df contains the entire dataset
print("Shape of the data:", df.shape)


Shape of the data: (9609797, 18)


In [2]:
# Show the shape of the data (rows, columns)
print("Shape of the data:", df.shape)

Shape of the data: (9609797, 18)


In [4]:
print(df.columns.tolist())

['Date received', 'Product', 'Sub-product', 'Issue', 'Sub-issue', 'Consumer complaint narrative', 'Company public response', 'Company', 'State', 'ZIP code', 'Tags', 'Consumer consent provided?', 'Submitted via', 'Date sent to company', 'Company response to consumer', 'Timely response?', 'Consumer disputed?', 'Complaint ID']


In [5]:
df.dtypes


Date received                   object
Product                         object
Sub-product                     object
Issue                           object
Sub-issue                       object
Consumer complaint narrative    object
Company public response         object
Company                         object
State                           object
ZIP code                        object
Tags                            object
Consumer consent provided?      object
Submitted via                   object
Date sent to company            object
Company response to consumer    object
Timely response?                object
Consumer disputed?              object
Complaint ID                     int64
dtype: object

In [8]:

# Count missing (null) values in each column
missing_values = df.isnull().sum().sort_values(ascending=False)

# Display results
print("Missing values per column:\n")
print(missing_values)

Missing values per column:

Tags                            8981029
Consumer disputed?              8841498
Consumer complaint narrative    6629041
Company public response         4770207
Consumer consent provided?      1649561
Sub-issue                        839522
Sub-product                      235295
State                             54516
ZIP code                          30228
Company response to consumer         20
Issue                                 6
Date received                         0
Product                               0
Company                               0
Date sent to company                  0
Submitted via                         0
Timely response?                      0
Complaint ID                          0
dtype: int64


In [9]:
missing_percent = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Values': df.isnull().sum(),
    'Percent Missing': missing_percent.round(2)
}).sort_values(by='Missing Values', ascending=False)

print(missing_df)


                              Missing Values  Percent Missing
Tags                                 8981029            93.46
Consumer disputed?                   8841498            92.01
Consumer complaint narrative         6629041            68.98
Company public response              4770207            49.64
Consumer consent provided?           1649561            17.17
Sub-issue                             839522             8.74
Sub-product                           235295             2.45
State                                  54516             0.57
ZIP code                               30228             0.31
Company response to consumer              20             0.00
Issue                                      6             0.00
Date received                              0             0.00
Product                                    0             0.00
Company                                    0             0.00
Date sent to company                       0             0.00
Submitte

In [11]:

# Count complaints per product
product_counts = df['Product'].value_counts()

# Display counts
print("Number of complaints per product:\n")
print(product_counts)

Number of complaints per product:

Product
Credit reporting or other personal consumer reports                             4834855
Credit reporting, credit repair services, or other personal consumer reports    2163857
Debt collection                                                                  799197
Mortgage                                                                         422254
Checking or savings account                                                      291178
Credit card                                                                      226686
Credit card or prepaid card                                                      206369
Money transfer, virtual currency, or money service                               145066
Credit reporting                                                                 140429
Student loan                                                                     109717
Bank account or service                                                      

In [15]:
df['Narrative_Word_Count'] = df['Consumer complaint narrative'].fillna('').apply(lambda x: len(str(x).split()))



# 3. Check extremes
short_narratives = df[df['Narrative_Word_Count'] < 5]
long_narratives = df[df['Narrative_Word_Count'] > 200]

print(f"\nVery short narratives (< 5 words): {short_narratives.shape[0]}")
print(f"Very long narratives (> 200 words): {long_narratives.shape[0]}")

# 4. Count complaints with and without narratives
with_narrative = df['Consumer complaint narrative'].notnull().sum()
without_narrative = df['Consumer complaint narrative'].isnull().sum()

print(f"\nComplaints with narratives: {with_narrative}")
print(f"Complaints without narratives: {without_narrative}")


Very short narratives (< 5 words): 6632216
Very long narratives (> 200 words): 789077

Complaints with narratives: 2980756
Complaints without narratives: 6629041


In [18]:

# Define keyword patterns and mapping to unified product categories
product_map = {
    "Credit card": "Credit Card",
    "Personal loan": "Personal Loan",
    "Buy Now, Pay Later": "BNPL",
    "BNPL": "BNPL",
    "Savings account": "Savings Account",
    "Money transfers": "Money Transfers",
    "Money transfer": "Money Transfers"
}

# Convert 'Product' to lowercase for easier matching
df['Product_clean'] = df['Product'].str.lower().fillna('')

# Function to assign standardized product category
def map_product(product_text):
    for keyword, new_category in product_map.items():
        if keyword.lower() in product_text:
            return new_category
    return None

# Apply mapping
df['Standard_Product'] = df['Product_clean'].apply(map_product)

# Filter only matched products
filtered_df = df[df['Standard_Product'].notnull()]

# Check result
print("Counts by standardized product:")
print(filtered_df['Standard_Product'].value_counts())

Counts by standardized product:
Standard_Product
Credit Card        433055
Savings Account    291178
Money Transfers    150420
Personal Loan       47155
Name: count, dtype: int64


In [20]:

# Define the product mapping
product_map = {
    "Credit card": "Credit Card",
    "Personal loan": "Personal Loan",
    "Buy Now, Pay Later": "BNPL",
    "BNPL": "BNPL",
    "Savings account": "Savings Account",
    "Money transfers": "Money Transfers",
    "Money transfer": "Money Transfers"
}

# Normalize product column
df['Product_clean'] = df['Product'].str.lower().fillna('')

# Map product to standard category
def map_product(product_text):
    for keyword in product_map:
        if keyword.lower() in product_text:
            return product_map[keyword]
    return None

# Create a new column with mapped values
df['Standard_Product'] = df['Product_clean'].apply(map_product)

# 🔍 Filter records not in the mapped product list
unmatched_df = df[df['Standard_Product'].isnull()]

# Count them
unmatched_count = unmatched_df.shape[0]
print(f"Number of complaints NOT in the 5 target product categories: {unmatched_count}")

# Optional: View top unmatched product types
print("\nTop unmatched product types:")
print(unmatched_df['Product'].value_counts().head(10))

Number of complaints NOT in the 5 target product categories: 8687989

Top unmatched product types:
Product
credit reporting or other personal consumer reports                             4834855
credit reporting, credit repair services, or other personal consumer reports    2163857
debt collection                                                                  799197
mortgage                                                                         422254
credit reporting                                                                 140429
student loan                                                                     109717
bank account or service                                                           86205
vehicle loan or lease                                                             72957
consumer loan                                                                     31574
prepaid card                                                                      15280
Name: count, 

In [23]:

# Define the five target product categories (as they appear in the dataset)
target_products = [
    "Credit card",
    "Personal loan",
    "Buy Now, Pay Later",
    "BNPL",
    "Savings account",
    "Money transfers"
]

# Filter the dataset
filtered_df = df[df['Product'].str.contains('|'.join(target_products), case=False, na=False)]

# Confirm shape and a preview
print(f"Filtered dataset shape: {filtered_df.shape}")
print(filtered_df['Product'].value_counts())

# Save the filtered dataset to a new CSV file
filtered_df.to_csv("../data/filtered.csv", index=False)
print("Filtered data saved to 'filtered_complaints.csv'")


Filtered dataset shape: (776742, 21)
Product
checking or savings account                                291178
credit card                                                226686
credit card or prepaid card                                206369
payday loan, title loan, or personal loan                   30641
payday loan, title loan, personal loan, or advance loan     16514
money transfers                                              5354
Name: count, dtype: int64
Filtered data saved to 'filtered_complaints.csv'
